# TAE-IA · Module 6 · L00 — Environment Setup

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Lesson** | L00 — Setup (run before L01) |
| **Estimated duration** | 10 minutes |
| **GPU required** | T4 recommended (not mandatory for this notebook) |

## What this notebook does

This is a one-time environment check that every student runs **before Lesson 1**. It does the following:

1. Mounts your Google Drive and creates the folder structure the entire course will use.
2. Verifies that your Colab runtime has a T4 GPU with enough VRAM.
3. Installs all Python packages needed across all 30 lessons in a single shot.
4. Sets up your HuggingFace token for gated models.
5. Configures environment variables and writes them to Drive so every subsequent notebook finds them.
6. Shows you how much Google Drive space the course will use.
7. Runs a fast smoke test (no model downloads) to confirm the environment is working.
8. Prints a pass/fail summary so you know whether you are ready to start L01.

Run every cell from top to bottom. The whole notebook should complete in about 10 minutes on the first run.

## Before you run this notebook you need

**(a) A Google account with Google Drive available.**  
You need at least 30 GB of free space on Drive. The course caches models and datasets there so they survive Colab session resets.

**(b) A HuggingFace account (free).**  
Go to [huggingface.co](https://huggingface.co) and create a free account. You will also need to generate a **read token** under `Settings > Access Tokens`. Some Track A models (Stable Diffusion) require accepting a license on HuggingFace before downloading; the setup notebook will tell you which ones.

**(c) Colab runtime set to T4 GPU.**  
Go to `Runtime > Change runtime type > T4 GPU` before running any cell. The notebook will warn you if this is missing, but it will not crash — some setup steps work without a GPU.

---

## Cell 1 — Mount Drive and create folder structure

This cell mounts your Google Drive and creates all directories the course needs. If a directory already exists, `exist_ok=True` means no error is raised — you can safely re-run this cell at any time.

In [ ]:
# Cell 1 — Mount Drive and create folder structure
import os

from google.colab import drive

if os.path.isdir('/content/drive/MyDrive'):
    print('Google Drive already mounted — skipping.')
else:
    drive.mount('/content/drive')
    print('Google Drive mounted.')

ROOT = '/content/drive/MyDrive/TAE_IA_M6'

DIRS = [
    ROOT,
    os.path.join(ROOT, 'models'),
    os.path.join(ROOT, 'models', 'hub'),
    os.path.join(ROOT, 'datasets'),
] + [os.path.join(ROOT, f'L{i:02d}_output') for i in range(1, 31)]

for d in DIRS:
    os.makedirs(d, exist_ok=True)

print('Folder structure verified.')
print(f'Course root : {ROOT}')
print()
print('Directories created or already present:')
for d in DIRS:
    label = d.replace(ROOT, 'TAE_IA_M6')
    print(f'  {label}')

## Cell 2 — GPU and system check

Verify that the Colab runtime has a GPU, identify it as a T4, and check VRAM. The course is designed for the T4 (16 GB VRAM). Some lessons with large models (SDXL-Turbo, BLIP-2) will fail on GPUs with less than 14 GB.

In [ ]:
# Cell 2 — GPU and system check
import sys
import torch

print('=== System Information ===')
print(f'Python      : {sys.version.split()[0]}')
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA built  : {torch.version.cuda}')
print()

gpu_ok   = False
vram_ok  = False
is_t4    = False

if not torch.cuda.is_available():
    print('WARNING: No GPU detected.')
    print('  Go to Runtime > Change runtime type > T4 GPU.')
    print('  You can continue setup, but lessons that run models will not work.')
else:
    gpu_ok   = True
    gpu_name = torch.cuda.get_device_name(0)
    props    = torch.cuda.get_device_properties(0)
    vram_gb  = props.total_memory / 1e9

    is_t4   = 'T4' in gpu_name
    vram_ok = vram_gb >= 14.0

    print('=== GPU Information ===')
    print(f'GPU name    : {gpu_name}')
    print(f'VRAM total  : {vram_gb:.1f} GB')
    print(f'CUDA version: {torch.version.cuda}')
    print()

    if not is_t4:
        print(f'WARNING: Expected T4, found "{gpu_name}".')
        print('  Most lessons target T4. Other GPUs may work, but are not tested.')
    else:
        print('GPU check passed: T4 detected.')

    if not vram_ok:
        print(f'WARNING: VRAM is {vram_gb:.1f} GB (minimum recommended: 14 GB).')
        print('  Lessons L07, L10, L12 may run out of memory.')
    else:
        print(f'VRAM check passed: {vram_gb:.1f} GB available.')

## Cell 3 — Package installation

Install all packages used across the entire 30-lesson course in one cell. This takes 4–6 minutes on a fresh Colab runtime. Packages are grouped by course track.

**Note on Coqui TTS:** The `TTS` package requires you to agree to the Coqui terms of service. The environment variable `COQUI_TOS_AGREED=1` is set here and again in Cell 5. Do not import `TTS` before this variable is set or you will see an interactive prompt that hangs Colab.

In [ ]:
# Cell 3 — Package installation (skips groups that are already installed)
import importlib
import importlib.util
import os
import subprocess
import sys

os.environ['COQUI_TOS_AGREED'] = '1'

def _is_installed(import_name):
    return importlib.util.find_spec(import_name) is not None

def _version(import_name):
    try:
        return importlib.import_module(import_name).__version__
    except Exception:
        return 'installed'

def _pip(packages_str, label):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + packages_str.split(),
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'  WARNING: install may have failed for {label}.')
        print(result.stderr[-500:] if result.stderr else '')

# --- PyTorch (pre-installed on Colab; check only) ---
if _is_installed('torch'):
    import torch
    print(f'[SKIP] torch already installed: {torch.__version__}')
else:
    print('[INSTALL] Installing PyTorch...')
    _pip('torch torchvision torchaudio', 'PyTorch')

# --- Track A — Vision ---
if _is_installed('diffusers') and _is_installed('controlnet_aux'):
    print(f'[SKIP] Track A packages already installed (diffusers {_version("diffusers")})')
else:
    print('[INSTALL] Installing Track A packages (diffusers, transformers, controlnet)...')
    _pip('diffusers transformers accelerate controlnet-aux invisible-watermark safetensors', 'Track A')

# --- Track B — Audio base (whisper, librosa) ---
if _is_installed('whisper') and _is_installed('librosa'):
    print(f'[SKIP] Track B base packages already installed (librosa {_version("librosa")})')
else:
    print('[INSTALL] Installing Track B base packages (whisper, librosa, soundfile)...')
    _pip('openai-whisper librosa soundfile', 'Track B base')

# --- Coqui TTS (community fork — supports Python 3.12) ---
# NOTE: the original PyPI 'TTS' package requires Python <3.11 and is abandoned.
# We use the Idiap community fork which is maintained and supports Python 3.12.
if _is_installed('TTS'):
    print(f'[SKIP] Coqui TTS already installed ({_version("TTS")})')
else:
    print('[INSTALL] Installing Coqui TTS (idiap fork, ~2 min)...')
    _pip('git+https://github.com/idiap/coqui-ai-TTS.git', 'Coqui TTS')

# --- Restoration (Real-ESRGAN) ---
if _is_installed('realesrgan'):
    print(f'[SKIP] Restoration packages already installed (realesrgan {_version("realesrgan")})')
else:
    print('[INSTALL] Installing restoration packages (realesrgan, basicsr, gfpgan)...')
    _pip('realesrgan basicsr facexlib gfpgan', 'restoration')

# --- Utilities ---
if _is_installed('gradio') and _is_installed('jiwer') and _is_installed('cv2'):
    print(f'[SKIP] Utility packages already installed (gradio {_version("gradio")})')
else:
    print('[INSTALL] Installing utility packages (gradio, jiwer, opencv, matplotlib)...')
    _pip('gradio jiwer gtts opencv-python matplotlib seaborn pandas Pillow requests', 'utilities')

# NOTE: L23 (MusicGen/AudioGen) uses HuggingFace transformers which is already
# installed above — no separate audiocraft package needed.

print()
print('=== Installed versions ===')
packages = [
    ('torch',        'torch'),
    ('diffusers',    'diffusers'),
    ('transformers', 'transformers'),
    ('accelerate',   'accelerate'),
    ('whisper',      'openai-whisper'),
    ('librosa',      'librosa'),
    ('soundfile',    'soundfile'),
    ('TTS',          'coqui-TTS (idiap)'),
    ('gradio',       'gradio'),
    ('jiwer',        'jiwer'),
    ('cv2',          'opencv-python'),
    ('matplotlib',   'matplotlib'),
    ('PIL',          'Pillow'),
]
all_ok = True
for imp, display in packages:
    if _is_installed(imp):
        print(f'  {display:<26} {_version(imp)}')
    else:
        print(f'  {display:<26} MISSING')
        all_ok = False

print()
if all_ok:
    print('All packages verified.')
else:
    print('WARNING: some packages are missing. Scroll up for install errors.')

## Cell 4 — HuggingFace token setup

Some Track A models require you to accept a license agreement on HuggingFace before downloading. The steps are:

1. Go to the model page on [huggingface.co](https://huggingface.co) (links are in each lesson notebook).
2. Click **Agree and access repository**.
3. Generate a **read token** under `Settings > Access Tokens`.
4. In Colab, open the **Secrets panel** (the key icon in the left sidebar), add a secret named `HF_TOKEN`, and paste your token as the value.

**Do not hardcode your token in the notebook.** Anyone who can see your notebook can see a hardcoded token.

If you do not provide a token now, the following lessons will not be able to download gated models:
- **L01, L02, L03, L04** — `runwayml/stable-diffusion-v1-5` (requires license acceptance)
- **L05** — ControlNet checkpoint
- **L08** — Style transfer variant

All other lessons use ungated models and will work without a token.

In [ ]:
# Cell 4 — HuggingFace token setup
import huggingface_hub

# Check if already logged in from a previous run
already_logged_in = False
try:
    existing = huggingface_hub.get_token()
    if existing:
        whoami = huggingface_hub.whoami()
        print(f'[SKIP] Already logged in to HuggingFace as: {whoami["name"]}')
        print('Gated models (SD 1.5, ControlNet) will be accessible.')
        already_logged_in = True
except Exception:
    pass

if not already_logged_in:
    HF_TOKEN = None

    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        if HF_TOKEN:
            print('HF_TOKEN found in Colab Secrets.')
        else:
            print('HF_TOKEN secret exists but is empty.')
    except Exception:
        print('Colab Secrets not accessible (expected outside Colab).')

    if HF_TOKEN:
        try:
            huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
            whoami = huggingface_hub.whoami()
            print(f'Logged in to HuggingFace as: {whoami["name"]}')
            print('Gated models (SD 1.5, ControlNet) will be accessible.')
        except Exception as e:
            print(f'WARNING: HuggingFace login failed: {e}')
            print('Check that your token is valid and has read permissions.')
    else:
        print()
        print('No HuggingFace token provided.')
        print('To add one:')
        print('  1. Open the Secrets panel (key icon in the left sidebar).')
        print('  2. Add a secret named HF_TOKEN with your token value.')
        print('  3. Re-run this cell.')
        print()
        print('Lessons affected without a token: L01, L02, L03, L04, L05, L08.')
        print('All other lessons will work normally.')

## Cell 5 — Environment variables

All course notebooks point model caches and HuggingFace downloads to your Google Drive. This prevents re-downloading models every time a Colab session resets.

The variables are set here and written to `env_setup.txt` on Drive so you can review them at any time.

In [ ]:
# Cell 5 — Environment variables
import os

MODEL_CACHE = '/content/drive/MyDrive/TAE_IA_M6/models'

env_vars = {
    'XDG_CACHE_HOME'    : MODEL_CACHE,
    'HF_HOME'           : MODEL_CACHE,
    'TORCH_HOME'        : MODEL_CACHE,
    'TRANSFORMERS_CACHE': MODEL_CACHE + '/hub',
    'COQUI_TOS_AGREED'  : '1',
}

for key, value in env_vars.items():
    os.environ[key] = value

print('Environment variables set:')
for key, value in env_vars.items():
    print(f'  {key:<22} = {value}')

# Write a reference file to Drive
env_file = '/content/drive/MyDrive/TAE_IA_M6/env_setup.txt'
lines = [
    'TAE-IA Module 6 — Environment Variables Reference',
    'Generated by M6_L00_setup.ipynb',
    '',
    '# Add these to each session or run M6_L00_setup.ipynb at the start of each session.',
    '# All course notebooks set these automatically in their setup cell.',
    '',
]
for key, value in env_vars.items():
    lines.append(f'{key}={value}')

with open(env_file, 'w') as f:
    f.write('\n'.join(lines) + '\n')

print()
print(f'Reference file written to: {env_file}')

## Cell 6 — Drive space estimate

Models are large. The table below shows the cumulative Drive space required as you progress through the course. Check that you have at least 30 GB free before starting.

| Category | Models / files | Approximate size |
|---|---|---|
| Track A models | SD 1.5, SDXL-Turbo, ControlNet, Real-ESRGAN, BLIP-2, CLIP | ~15 GB |
| Track B models | Whisper small/medium, XTTS-v2, MusicGen-small, AudioGen-medium | ~8 GB |
| Datasets | ESC-50 (environmental sounds) | ~0.6 GB |
| Lesson outputs | Images, audio, plots from 30 lessons | ~2 GB |
| **Total** | | **~26 GB** |

Models are downloaded on demand — you will not use 26 GB on Day 1. If you only complete Track A or Track B, the total will be closer to 17–18 GB.

In [ ]:
# Cell 6 — Drive space estimate and free-space check
import shutil

DRIVE_PATH = '/content/drive/MyDrive'
RECOMMENDED_FREE_GB = 30.0

try:
    total, used, free = shutil.disk_usage(DRIVE_PATH)
    total_gb = total / 1e9
    used_gb  = used  / 1e9
    free_gb  = free  / 1e9

    print('=== Google Drive Space ===')
    print(f'  Total capacity : {total_gb:.1f} GB')
    print(f'  Used           : {used_gb:.1f} GB')
    print(f'  Free           : {free_gb:.1f} GB')
    print()

    if free_gb < RECOMMENDED_FREE_GB:
        print(f'WARNING: Only {free_gb:.1f} GB free on Drive.')
        print(f'  Recommended minimum: {RECOMMENDED_FREE_GB:.0f} GB.')
        print('  Free up space before downloading large models, or some lessons may fail.')
    else:
        print(f'Drive space check passed: {free_gb:.1f} GB free (minimum {RECOMMENDED_FREE_GB:.0f} GB).')

except Exception as e:
    print(f'Could not read Drive space: {e}')
    print('Make sure Google Drive is mounted (Cell 1) before running this cell.')

print()
print('=== Cumulative Course Space by Track ===')
rows = [
    ('Track A models', 'SD 1.5, SDXL-Turbo, ControlNet, Real-ESRGAN, BLIP-2, CLIP', '~15 GB'),
    ('Track B models', 'Whisper small/medium, XTTS-v2, MusicGen-small, AudioGen-medium', '~8 GB'),
    ('Datasets',       'ESC-50 (environmental sounds)', '~0.6 GB'),
    ('Lesson outputs', 'Images, audio, plots from 30 lessons', '~2 GB'),
    ('TOTAL',          '', '~26 GB'),
]
print(f'  {"Category":<20} {"Contents":<55} {"Size"}')
print('  ' + '-' * 82)
for cat, contents, size in rows:
    print(f'  {cat:<20} {contents:<55} {size}')

## Cell 7 — Quick smoke test

Confirm the environment works by importing one representative package per track and running a tiny computation. No models are downloaded here.

In [ ]:
# Cell 7 — Quick smoke test (no model downloads)
import importlib
import numpy as np

results = {}

# Track A — Vision: diffusers
try:
    import diffusers
    print(f'diffusers   : {diffusers.__version__}  OK')
    results['diffusers'] = True
except ImportError as e:
    print(f'diffusers   : FAILED ({e})')
    results['diffusers'] = False

# Track B — Audio: librosa
try:
    import librosa
    print(f'librosa     : {librosa.__version__}  OK')
    results['librosa'] = True
except ImportError as e:
    print(f'librosa     : FAILED ({e})')
    results['librosa'] = False

# Utilities: gradio
try:
    import gradio
    print(f'gradio      : {gradio.__version__}  OK')
    results['gradio'] = True
except ImportError as e:
    print(f'gradio      : FAILED ({e})')
    results['gradio'] = False

# Numeric sanity check — 0.5-second sine wave at 22050 Hz
sample_rate = 22050
duration    = 0.5
frequency   = 440.0  # A4
t           = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
sine_wave   = np.sin(2 * np.pi * frequency * t)

expected_samples = int(sample_rate * duration)  # 11025
if sine_wave.shape == (expected_samples,):
    print(f'sine wave   : shape {sine_wave.shape}  range [{sine_wave.min():.3f}, {sine_wave.max():.3f}]  OK')
    results['numpy_audio'] = True
else:
    print(f'sine wave   : FAILED — unexpected shape {sine_wave.shape}')
    results['numpy_audio'] = False

print()
if all(results.values()):
    print('Smoke test passed: all imports and basic computation succeeded.')
else:
    failed = [k for k, v in results.items() if not v]
    print(f'Smoke test WARNING: {len(failed)} item(s) failed: {failed}')
    print('Review the output above and re-run Cell 3 if packages are missing.')

## Cell 8 — Final checklist

This cell re-runs all checks silently and prints a pass/fail summary. If everything passes, you are ready to open L01.

In [ ]:
# Cell 8 — Final checklist
import importlib
import os
import shutil
import sys

PASS = 'PASS'
FAIL = 'FAIL'
WARN = 'WARN'

checks = []  # list of (label, status, detail)

# --- 1. Drive mounted ---
drive_root = '/content/drive/MyDrive/TAE_IA_M6'
if os.path.isdir(drive_root):
    checks.append(('Google Drive mounted', PASS, drive_root))
else:
    checks.append(('Google Drive mounted', FAIL, 'Run Cell 1'))

# --- 2. Folder structure ---
required_dirs = [
    os.path.join(drive_root, 'models'),
    os.path.join(drive_root, 'models', 'hub'),
    os.path.join(drive_root, 'datasets'),
    os.path.join(drive_root, 'L01_output'),
    os.path.join(drive_root, 'L30_output'),
]
missing_dirs = [d for d in required_dirs if not os.path.isdir(d)]
if not missing_dirs:
    checks.append(('Folder structure', PASS, 'All directories present'))
else:
    checks.append(('Folder structure', FAIL, f'{len(missing_dirs)} directories missing — re-run Cell 1'))

# --- 3. GPU ---
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
        if 'T4' in gpu_name and vram_gb >= 14.0:
            checks.append(('GPU (T4, >=14 GB)', PASS, f'{gpu_name}, {vram_gb:.1f} GB'))
        elif vram_gb < 14.0:
            checks.append(('GPU (T4, >=14 GB)', WARN, f'{gpu_name}, {vram_gb:.1f} GB — below 14 GB'))
        else:
            checks.append(('GPU (T4, >=14 GB)', WARN, f'{gpu_name} is not T4 — lessons untested on this GPU'))
    else:
        checks.append(('GPU (T4, >=14 GB)', FAIL, 'No GPU — go to Runtime > Change runtime type > T4 GPU'))
except ImportError:
    checks.append(('GPU (T4, >=14 GB)', FAIL, 'torch not importable'))

# --- 4. Key packages ---
required_packages = [
    ('diffusers',    'diffusers'),
    ('transformers', 'transformers'),
    ('whisper',      'openai-whisper'),
    ('librosa',      'librosa'),
    ('gradio',       'gradio'),
    ('cv2',          'opencv-python'),
]
pkg_missing = []
for imp, display in required_packages:
    if importlib.util.find_spec(imp) is None:
        pkg_missing.append(display)
if not pkg_missing:
    checks.append(('Packages installed', PASS, 'All key packages present'))
else:
    checks.append(('Packages installed', FAIL, f'Missing: {pkg_missing} — re-run Cell 3'))

# --- 5. HuggingFace token ---
try:
    import huggingface_hub
    token_info = huggingface_hub.get_token()
    if token_info:
        checks.append(('HuggingFace token', PASS, 'Token present and cached'))
    else:
        checks.append(('HuggingFace token', WARN, 'No token — L01–L05, L08 require gated model access'))
except Exception:
    checks.append(('HuggingFace token', WARN, 'Could not verify — run Cell 4 to log in'))

# --- 6. Environment variables ---
hf_home = os.environ.get('HF_HOME', '')
if hf_home and 'TAE_IA_M6' in hf_home:
    checks.append(('Env vars (HF_HOME)', PASS, hf_home))
else:
    checks.append(('Env vars (HF_HOME)', FAIL, 'HF_HOME not set — re-run Cell 5'))

# --- 7. Drive space ---
try:
    _, _, free = shutil.disk_usage('/content/drive/MyDrive')
    free_gb = free / 1e9
    if free_gb >= 30.0:
        checks.append(('Drive free space', PASS, f'{free_gb:.1f} GB free'))
    else:
        checks.append(('Drive free space', WARN, f'{free_gb:.1f} GB free (recommended: 30 GB)'))
except Exception:
    checks.append(('Drive free space', WARN, 'Could not read — verify manually'))

# --- Print summary ---
print('=' * 60)
print('  L00 Setup Checklist')
print('=' * 60)
for label, status, detail in checks:
    marker = '[  OK  ]' if status == PASS else ('[ WARN ]' if status == WARN else '[ FAIL ]')
    print(f'{marker}  {label}')
    if status != PASS:
        print(f'           {detail}')
print('=' * 60)

failures = [label for label, status, _ in checks if status == FAIL]
warnings = [label for label, status, _ in checks if status == WARN]

print()
if not failures:
    if not warnings:
        print('READY -- you can start L01.')
    else:
        print('READY WITH WARNINGS -- you can start L01.')
        print(f'Address these warnings before the affected lessons:')
        for w in warnings:
            print(f'  - {w}')
else:
    print('NOT READY -- fix the following before starting L01:')
    for f in failures:
        print(f'  - {f}')
    if warnings:
        print()
        print('Also review these warnings:')
        for w in warnings:
            print(f'  - {w}')

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L00*  
*Platform: Google Colab (T4 GPU) · Python 3.10*